# Minimal SPH (Swift)
## Sedov Blast test for Peano SPH


In this notebook, we set up the Sedov-Taylor Blast test, which is part of a battery of well-established tests for SPH codes.

In this test, particles are initialised at rest in regular grid configuration and periodic BC. Gas has negligible presure. Energy ($E$) is injected to a (or group of) central particle and the blast should propagate isotropically, up to boundary effects. See, e.g., Springel and Hernquist, Mon. Not. R. Astron. Soc. 333, 649–664 (2002).

## Particle behaviour

As usual, in peano 4 the particle-particle interactions are handled in terms of the active and local particle sets. Notice that in this we now do not simply rely on the particles of the adjacent vertices of a cell -- see the sketch below. To understand what the individual particles in the sketch are supposed to mean and how they are maintained and exposed to the user, please consult the documentation of the ParticleParticleInteraction class in the Python toolbox directory. It contains a lot explanations.

**Remarks**:
- For particle-particle interactions: bear in mind that the local set is a subset of the active set, so be careful about any unwanted self-interactions.
- In this scheme, the following relation should be satisfied:

\begin{align}
H_{\rm SPH} \leq R^{\rm peano}_{\rm cutoff} \leq \frac{\Delta x}{2}\,,
\end{align}

where $H_{\rm SPH}$($\equiv\gamma_K h$) is the SPH interaction radius, $R^{\rm peano}_{\rm cutoff}$ is the internal cut-off radius and $\Delta x$ the cell size.

In [1]:
import os

import peano4
import peano4.dastgen2
import peano4.toolbox
import peano4.toolbox.particles
import dastgen2

DaStGen 2 (C) www.peano-framework.org
Peano 4 (C) www.peano-framework.org


In [2]:
# If we want to start a project from scratch, we may want to remove some of the generated code
!make clean; rm -r actions/ vertexdata/ repositories/ globaldata/ celldata/ observers/ 

project = peano4.Project( ["examples", "particles"], "minimal-sph-peano-test-sedov", "." )

rm -f celldata/ParticleSetCellStatistics.o vertexdata/ParticleSet.o globaldata/Particle.o repositories/DataRepository.o ./observers/CreateGrid2peano4_toolbox_CreateRegularGrid0.o ./observers/CreateGrid.o ./observers/Init2peano4_toolbox_particles_UpdateParticleGridAssociation0.o ./observers/Init2peano4_toolbox_particles_InsertParticlesAlongCartesianLayoutIntoUnrefinedCells1.o ./observers/Init2peano4_toolbox_particles_ParticleTreeAnalysis2.o ./observers/Init.o ./actions/DensityNonSym.o ./observers/DensityNonSym2peano4_toolbox_particles_UpdateParticleGridAssociation1.o ./observers/DensityNonSym2peano4_toolbox_particles_ParticleParticleInteraction2.o ./observers/DensityNonSym2peano4_toolbox_particles_ParticleAMR3.o ./observers/DensityNonSym2peano4_toolbox_particles_ParticleTreeAnalysis4.o ./observers/DensityNonSym.o ./actions/NewtonRaphsonSmoothingLength.o ./observers/NewtonRaphsonSmoothingLength2peano4_toolbox_particles_UpdateParticleGridAssociation1.o ./observers/NewtonRaphsonSmoothingLe

## Prepare code

Peano's API does not know which settings to use on the present system. To make it copy/clone the settings identified by ./configure, we ask it to parse the generated configuration scripts. 

In [3]:
build_mode = peano4.output.CompileMode.Asserts
#build_mode = peano4.output.CompileMode.Release
project.output.makefile.parse_configure_script_outcome( "../../.." )
project.output.makefile.set_mode( build_mode )

parse configure outcome ../../../config.log to extract configure settings
found the configure call info   $ ./configure --with-mpi=mpicxx --with-multithreading=omp CXXFLAGS=--std=c++17 -DnoMPISupportsSingleSidedCommunication -fopenmp -funroll-loops CC=gcc CXX=g++ FC=gfortran --enable-loadbalancing --enable-exahype --enable-particles --enable-blockstructured --no-create --no-recursion

parse configure outcome ../../../src/Makefile to extract compile settings
add CXX=mpicxx
add CXXFLAGS=--std=c++17 -DnoMPISupportsSingleSidedCommunication -fopenmp -funroll-loops
add CXXFLAGS_PEANO_2D_ASSERTS=-DDimensions=2 -DPeanoDebug=2 -g3 -O0
add CXXFLAGS_PEANO_2D_DEBUG=-DDimensions=2 -DPeanoDebug=4 -g3 -O0
add CXXFLAGS_PEANO_2D_RELEASE=-DDimensions=2 -DPeanoDebug=0
add CXXFLAGS_PEANO_2D_STATS=-DDimensions=2 -DPeanoDebug=0 -DTrackStatistics
add CXXFLAGS_PEANO_2D_TRACE=-DDimensions=2 -DPeanoDebug=1 -g3
add CXXFLAGS_PEANO_3D_ASSERTS=-DDimensions=3 -DPeanoDebug=2 -g3 -O0
add CXXFLAGS_PEANO_3D_DEBUG=-DDime

## Model the particle

We create the particle through the toolbox. A particle by definition has a position, so we just need to add everything else for SPH:
- Velocity array (${\bf v}$)
- Acceleration array (${\bf a}\equiv{\rm d}{\bf v}/{\rm d}t$)
- Mass
- Density
- Pressure
- Internal Energy ($u$)
- Time derivative of internal energy (${\rm d}u/{\rm d}t$)
- Kernel smoothing length ($h$)

We add a marker to indicate the next calculation step required for a particle so we don't perform the same step on a particle twice in a row.

We also add an array to store the new position of a particle after a timestep is completed. We will use this to move the particle at the start of the subsequent timestep.

In [4]:
particle = peano4.toolbox.particles.Particle("Particle")

# Define properties
v_attr = peano4.dastgen2.Peano4DoubleArray("v","Dimensions")
a_attr = peano4.dastgen2.Peano4DoubleArray("a","Dimensions")
a_prev_step_attr = peano4.dastgen2.Peano4DoubleArray("a_prev_step","Dimensions")
mass_attr    = dastgen2.attributes.Double("mass")
smoothL_attr = dastgen2.attributes.Double("smoothingLength")
density_attr = dastgen2.attributes.Double("density")
pressure_attr= dastgen2.attributes.Double("pressure")
u_attr       = dastgen2.attributes.Double("u")
uDot_attr    = dastgen2.attributes.Double("uDot")
uDot_prev_step_attr  = dastgen2.attributes.Double("uDot_prev_step")
soundSpeed_attr = dastgen2.attributes.Double("soundSpeed")
wcount_attr   = dastgen2.attributes.Double("wcount")
wcount_dh_attr= dastgen2.attributes.Double("wcount_dh")
Ncount_attr   = dastgen2.attributes.Integer("Ncount")

# Variable smoothing length scheme
Omega_attr = dastgen2.attributes.Double("Omega")
rho_dh_attr= dastgen2.attributes.Double("rho_dh")
hDot_attr  = dastgen2.attributes.Double("hDot")

# Artificial viscosity scheme
#rot_v_attr = peano4.dastgen2.Peano4DoubleArray("rot_v","Dimensions")
rot_v_attr    = dastgen2.attributes.Double("rot_v")    # For 2D sims it only has 1 component (z)
div_v_attr    = dastgen2.attributes.Double("div_v")
v_sig_AV_attr = dastgen2.attributes.Double("v_sig_AV")

balsara_attr  = dastgen2.attributes.Double("balsara")

# Needed?
has_been_kicked_attr = dastgen2.attributes.Boolean("hasBeenKicked")
has_converged_attr= dastgen2.attributes.Boolean("hasConverged")
smoothL_prev_attr = dastgen2.attributes.Double("smoothingLengthPrevious")

# Debug
time_attr     = dastgen2.attributes.Double("time")
timeStepSize_attr = dastgen2.attributes.Double("timeStepSize")

# Add particle properties
particle.data.add_attribute(v_attr)
particle.data.add_attribute(a_attr)
particle.data.add_attribute(a_prev_step_attr)
particle.data.add_attribute(mass_attr)
particle.data.add_attribute(density_attr)
particle.data.add_attribute(pressure_attr)
particle.data.add_attribute(u_attr)
particle.data.add_attribute(uDot_attr)
particle.data.add_attribute(uDot_prev_step_attr)
particle.data.add_attribute(smoothL_attr)
particle.data.add_attribute(Ncount_attr)
particle.data.add_attribute(wcount_attr)
particle.data.add_attribute(wcount_dh_attr)
particle.data.add_attribute(soundSpeed_attr)

# For variable smoothing length
particle.data.add_attribute(Omega_attr)
particle.data.add_attribute(rho_dh_attr)
particle.data.add_attribute(hDot_attr)

# Artificial viscosity
particle.data.add_attribute(rot_v_attr)
particle.data.add_attribute(div_v_attr)
particle.data.add_attribute(v_sig_AV_attr)
particle.data.add_attribute(balsara_attr)

particle.data.add_attribute(time_attr)
particle.data.add_attribute(timeStepSize_attr)

# Needed?
particle.data.add_attribute(has_been_kicked_attr)
particle.data.add_attribute(has_converged_attr)
particle.data.add_attribute(smoothL_prev_attr)

particles = peano4.toolbox.particles.ParticleSet(particle)

This particle has to be added to the project. This way the project knows that we are working with this guy. We associate the particle with the vertices, as we work with the pidt (particle in dual tree) algorithm. 

In [5]:
project.datamodel.add_global_object(particle)
project.datamodel.add_vertex       (particles)

## Export global SPH constants and parameters

There are plenty of constant simulation paramaters to specify for our SPH simulation - we do that here. 

In [6]:
Dimensions = 2
project.output.makefile.set_dimension(Dimensions)

In [7]:
from numpy import pi

GLOBAL_TIME_STEP_SIZE = "1e-5"

# Maximum allowed number of Newton-Raphson iterations
HYDRO_DEFAULT_MAX_ITERATIONS = "1"

# Time-integration parameters
CFL_CONSTANT = "0.2"

# EoS parameter
GAMMA = str(5./3.)

# SPH kernel parameters
KERNEL_SUPPORT_RAD = "2"  # "gamma" parameter. =2 for Cubic Spline. Different in other kernels.

# Artificial viscosity parameters
ALPHA_AV= "0.8"
BETA_AV = "3"       # =3 in SPHENIX

# Smoothing-length free parameter (resolution "eta_neighbours"). 
# Can be set by specifying a desired number of neighbours.
if(Dimensions==2):
#    N_ngbs_mean = 10.0  
#    SMOOTH_L_FAC = str( (1.   /pi*N_ngbs_mean)**(1./2.)/float(KERNEL_SUPPORT_RAD) ) 
# SWIFT value. Should be equivalent to N_ngbs_mean = 15.14.   
    SMOOTH_L_FAC = str( 1.235 ) 
elif(Dimensions==3):
    N_ngbs_mean = 57.9
    SMOOTH_L_FAC = str( (3./4./pi*N_ngbs_mean)**(1./3.)/float(KERNEL_SUPPORT_RAD) ) 

# Kernel root value 
if(Dimensions==2):
    CUBIC_SPLINE_SIGMA = str(10/(7*pi))
elif(Dimensions==3):
    CUBIC_SPLINE_SIGMA = str(1/pi)

print("Smoothing length free factor: " + SMOOTH_L_FAC)
print("Kernel Root value (sigma): " + CUBIC_SPLINE_SIGMA)

# parameters specific to a given problem (e.g., Sedov blast, Sod wave, ...)
BLAST_CENTER = "0.5"    # {0.5,0.5}
#BLAST_ENERGY = "0.02"
BLAST_ENERGY = "1.0"
SPIKE_RADIUS = "0.02"

project.constants.export_const_with_type("GLOBAL_TIME_STEP_SIZE", GLOBAL_TIME_STEP_SIZE, "double")
project.constants.export_const_with_type("HYDRO_DEFAULT_MAX_ITERATIONS", HYDRO_DEFAULT_MAX_ITERATIONS, "int")
project.constants.export_const_with_type("KERNEL_SUPPORT_RAD"   , KERNEL_SUPPORT_RAD   , "double")
project.constants.export_const_with_type("SMOOTH_L_FAC"         , SMOOTH_L_FAC         , "double")
project.constants.export_const_with_type("CUBIC_SPLINE_SIGMA"   , CUBIC_SPLINE_SIGMA   , "double")
project.constants.export_const_with_type("BLAST_CENTER", BLAST_CENTER, "double")
project.constants.export_const_with_type("BLAST_ENERGY", BLAST_ENERGY, "double")
project.constants.export_const_with_type("SPIKE_RADIUS", SPIKE_RADIUS, "double")
project.constants.export_const_with_type("CFL_CONSTANT", CFL_CONSTANT, "double")
project.constants.export_const_with_type("GAMMA"       , GAMMA       , "double")
project.constants.export_const_with_type("ALPHA_AV" , ALPHA_AV , "double")
project.constants.export_const_with_type("BETA_AV"  , BETA_AV  , "double")

Smoothing length free factor: 1.235
Kernel Root value (sigma): 0.4547284088339867


## Model the setup phases

## Initialize regular grid

In [8]:
mesh_size = 0.5

particle_tree_analysis = peano4.toolbox.particles.ParticleTreeAnalysis(particles)
project.datamodel.add_cell(particle_tree_analysis.cell_marker)   # read docu of ParticleTreeAnalysis

create_grid = peano4.solversteps.Step( name="CreateGrid", add_user_defined_actions=False )
create_grid.add_action_set( peano4.toolbox.CreateRegularGrid(mesh_size) )
create_grid.use_vertex(particles)
create_grid.use_cell(particle_tree_analysis.cell_marker)
project.solversteps.add_step(create_grid)

## Initial conditions for particles

In [9]:
initialisation_snippet = """
 
  /*
   * Initial Conditions for the particular SPH test
   * 
   * 1. Sedov Blast: particles are initialised at rest in regular grid configuration and periodic BC. 
   * Energy is injected to a (or group of) central particle and the blast should propagate
   * isotropically, up to boundary effects.
   * See, e.g., Springel and Hernquist, Mon. Not. R. Astron. Soc. 333, 649–664 (2002).
   *
   */
   
  // Internal cutoff radious used by Peano. Should always be larger than actual SPH cutoff radious (2h).
  particle->setCutOffRadius(0.16);
  
  particle->setDensity ( 1.0 );        
 // particle->setMass    ( particle->getDensity()/400/2.56069 );       
  particle->setMass    ( particle->getDensity()/400/2.56069 );       
  particle->setPressure( 1e-6);
  
  particle->setSmoothingLength( SMOOTH_L_FAC * std::pow(particle->getMass()/particle->getDensity(), 1.0/Dimensions) );
//  particle->setSmoothingLength( 0.075 );
  particle->setSmoothingLengthPrevious( 0.99 * particle->getSmoothingLength() );
 
  // Initialise the blast
  tarch::la::Vector<Dimensions,double> blast_cent(BLAST_CENTER);
  const double dist_blast = tarch::la::norm2( particle->getX() - blast_cent );
  
 // /* Old method */
 // /* Inject energy evenly into central particles */
 // if ( tarch::la::smaller( dist_blast, SPIKE_RADIUS ) ) {
 //   particle->setU( BLAST_ENERGY );
 
   /* New method */
   /* Smooth the explosion with kernel */
   const double h = particle->getSmoothingLength();
   const double R_smooth = 4.0 * h;
   
   // Evaluate kernel
   const double q = dist_blast / h;
   double w, dw_dq;
   kernel_deval(q, w, dw_dq);
 
   // Normalization factors 
   const double W = w / std::pow( h, Dimensions );
 
  if ( tarch::la::smallerEquals( dist_blast, R_smooth ) ) {
    particle->setU( BLAST_ENERGY * W ) ;
  } else {
    particle->setU( 0.0 ); 
  }
  
  /* Set non-zero (background) value in order to get non-zero sound speed across the whole domain */
  if( tarch::la::equals(particle->getU(),0) ){
    particle->setU( particle->getPressure()/( (GAMMA-1) * particle->getDensity() ) ); 
  }
  
  // Evaluate (global) time-step size using u from exploding particle (h_ini is homogeneous)
  const double soundspeed = std::sqrt( GAMMA * particle->getPressure()/particle->getDensity() );
  particle->setSoundSpeed(soundspeed);
  
  
  const double timeStepSizeCFL = CFL_CONSTANT * particle->getSmoothingLength()*KERNEL_SUPPORT_RAD/soundspeed;
  
  particle->setTimeStepSize(GLOBAL_TIME_STEP_SIZE);
 
 /*
  if ( tarch::la::smaller( timeStepSizeCFL , GLOBAL_TIME_STEP_SIZE) ){
    particle->setTimeStepSize(timeStepSizeCFL);
  }
 */
 
  particle->setHasBeenKicked(false);
 
  // Set remaining fields to zero
  
  // Vectors
  for(int d=0; d<Dimensions; d++){
     particle->setV          (d,0);
     particle->setA          (d,0);
     particle->setA_prev_step(d,0);
  }
 
  /* For 2D sims, we only get one component */
  particle->setRot_v(0);
  
  // Initialize the remaining fields to zero
  particle->setUDot  (0.0);        
  particle->setUDot_prev_step(0.0);        
  particle->setHDot  (0.0);        
  particle->setWcount(0.0);        
  particle->setNcount(0  );        
  particle->setOmega (0.0);        
  particle->setRho_dh(0.0);           
  particle->setWcount_dh(0.0);  
  
  particle->setHasConverged(false);
  particle->setTime(0.0);
 
  
  // Artificial viscosity scheme
  particle->setDiv_v   (0.0);
  particle->setV_sig_AV(0.0);
  particle->setBalsara(0.0);
 
 
"""

init_setup = peano4.solversteps.Step( name="Init", add_user_defined_actions=False )
init_setup.use_vertex(particles)
init_setup.use_cell(particle_tree_analysis.cell_marker)
init_setup.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
init_setup.add_action_set( peano4.toolbox.particles.InsertParticlesAlongCartesianLayoutIntoUnrefinedCells(
  particle_set=particles,
  distance_between_particles=mesh_size/16,
#  distance_between_particles=mesh_size/60,
  initialisation_call=initialisation_snippet,
  noise=False,
  additional_includes=""" 
    #include "Constants.h" 
    #include "kernel_hydro.h"
    """,
  computational_domain_offset=[0.0, 0.0])
  )

init_setup.add_action_set( particle_tree_analysis )
project.solversteps.add_step(init_setup)

# Model the steps of the simulation

In the following, we model each step that will be invoked by main.cpp to run the simulation. The generated C++ code based on these will be outputted into `/observers/`

## Density loop

In [10]:
calc_densities_var_h = peano4.solversteps.Step( "DensityNonSym" )
calc_densities_var_h.use_vertex(particles)
calc_densities_var_h.use_cell(particle_tree_analysis.cell_marker)
calc_densities_var_h.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )

particle_particle_compute_kernel = """

 /*
  * Non-symmetric density calculation (as per Swift terminology). 
  * In this, only the Local particle is updated (in the symmetric case, both are updated at the same time).
  *
  *
  * Here we calculate the density, Omega and drho/dt sums.
  * The latter can be used in the Newton-Raphson iteration.
  *
  * Subscripts a and b represent indices of Local and Active particles, respectively:
  *
  * h_a: Smoothing length of Local particle
  * h_b: Smoothign length of Active particle
  * dx : Vector separating the two particles (x_loc - x_act)
  * r  : Distance between the two particles  (|| dx ||)
  *
  */


  tarch::la::Vector<Dimensions,double> dx = localParticle->getX() - activeParticle->getX();
  const double r = tarch::la::norm2(dx);

  // Velocity terms for viscosity
  tarch::la::Vector<Dimensions,double> dv_ab;
  tarch::la::Vector<Dimensions,double> curlvr(0.0);

  // Retrieve basic properties of local and active particles
  const double rho_b = activeParticle->getDensity();

  const double m_a = localParticle ->getMass();  
  const double m_b = activeParticle->getMass();  

  const double h_a = localParticle->getSmoothingLength();

  // Normalized distance to evaluate kernel
  const double q_a = r / h_a;
  
  // Get actual interaction radius
  const double cutoffR = KERNEL_SUPPORT_RAD*h_a;
  
  if ( tarch::la::smaller( r,cutoffR ) ) {
    if ( tarch::la::greater( r,0.0 ) ) {
     
      // Evaluate kernel and its derivative
      double w_a, dw_a_dq;
      kernel_deval(q_a, w_a, dw_a_dq);

      // Normalization factors 
      double W_a, dW_a_dh;
      W_a = w_a / std::pow( h_a, Dimensions );
      dW_a_dh = - ( Dimensions*w_a + q_a*dw_a_dq ) / std::pow( h_a, Dimensions +1 );

      // Increment density of Local particle
      localParticle->setDensity( localParticle-> getDensity() + m_b * W_a ); 

      // Increment number density as well
      localParticle->setWcount ( localParticle-> getWcount () +       W_a );
      
      // Increment exact number of neighbours (for debugging)
      localParticle->setNcount ( localParticle-> getNcount () + 1 ); 

      // Increment drho/dh
      localParticle->setRho_dh   ( localParticle->getRho_dh()    + m_b*dW_a_dh ); 

      // Increment Omega of Local particle
      localParticle->setOmega    ( localParticle->getOmega()     + m_b*dW_a_dh );

      // Increment dWcount/dh
      localParticle->setWcount_dh( localParticle->getWcount_dh() +     dW_a_dh ); 

      assertion3( not std::isnan( localParticle->getOmega() ), localParticle->toString(), localParticle->getOmega(), dW_a_dh );
      
      /* Now we need to compute the div terms */
          
      const double r_inv = r ? 1.0 / r : 0.0;     
      const double fac_a = m_a * dw_a_dq * r_inv;

      /* Compute dv dot r */
      dv_ab = localParticle->getV() - activeParticle->getV();
      const double dvdr = tarch::la::dot( dv_ab , dx );
      localParticle->setDiv_v( localParticle->getDiv_v() - fac_a * dvdr );
      
      /* Compute dv cross r */
      /* 2D Term only has 1 component (z) */
      const double curlvr2D = dv_ab[0]*dx[1] - dv_ab[1]*dx[0];
      localParticle->setRot_v( localParticle->getRot_v() + fac_a * curlvr2D );

    } 
  }
    
"""

touch_particle_last_time_compute_kernel = """

  /* Finalize the calculations from the density loop */

  if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){

    /* Retrieve some properties */
    const double m_a = localParticle->getMass();
    const double h_a = localParticle->getSmoothingLength();
    
    const double h_inv_dim = 1.0/std::pow( h_a, Dimensions );
    const double h_inv_dim_plus_one = h_inv_dim/h_a;

    // Default SPH kernel (cubic spline)
    double W_a = CUBIC_SPLINE_SIGMA ; // kernel form taken at q=0
      
    /* Add the contributions due to the particle itself (i.e. at q=0) */

    localParticle->setDensity  ( localParticle->getDensity()   + m_a * W_a * h_inv_dim );                      
    localParticle->setWcount   ( localParticle->getWcount()    +       W_a * h_inv_dim );                      
    localParticle->setOmega    ( localParticle->getOmega()     + m_a * (-Dimensions * W_a * h_inv_dim_plus_one) );   
    localParticle->setRho_dh   ( localParticle->getRho_dh()    + m_a * (-Dimensions * W_a * h_inv_dim_plus_one) );   
    localParticle->setWcount_dh( localParticle->getWcount_dh() +       (-Dimensions * W_a * h_inv_dim_plus_one) );
    localParticle->setNcount   ( localParticle->getNcount()    + 1 );

    const double rho_inv = 1.0 / localParticle->getDensity();

    /* Finish calculation of the velocity curl components */
    localParticle->setRot_v( localParticle->getRot_v() * h_inv_dim_plus_one * rho_inv);

    /* Finish calculation of the velocity divergence */
    localParticle->setDiv_v( localParticle->getDiv_v() * h_inv_dim_plus_one * rho_inv);
  
  }
  
  
"""

touch_particle_first_time_compute_kernel = """

  if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){

    // we'll accumulate the data throughout mesh/particle traversal
    localParticle->setDensity (0.0);                      
    localParticle->setWcount  (0.0);                      
    localParticle->setOmega   (0.0);   
    localParticle->setRho_dh  (0.0);   
    localParticle->setWcount_dh(0.0);   
    localParticle->setPressure(0.0);  
    localParticle->setDiv_v   (0.0);   
    localParticle->setRot_v   (0.0);   
    localParticle->setNcount  (0);
    
    // Also mark this for later Newton-Raphson iteration
    localParticle->setHasConverged(false);

  } 
    

"""

calc_densities_var_h.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel=particle_particle_compute_kernel,
  touch_vertex_first_time_compute_particle_update_kernel=touch_particle_first_time_compute_kernel,
  touch_vertex_last_time_compute_particle_update_kernel=touch_particle_last_time_compute_kernel,
  additional_includes="""
#include "Constants.h"
#include "kernel_hydro.h"
  """
))
calc_densities_var_h.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
calc_densities_var_h.add_action_set( particle_tree_analysis )
project.solversteps.add_step(calc_densities_var_h)

## Newton-Raphson

In [11]:
calc_newton_raphson_h = peano4.solversteps.Step( "NewtonRaphsonSmoothingLength" )
calc_newton_raphson_h.use_vertex(particles)
calc_newton_raphson_h.use_cell(particle_tree_analysis.cell_marker)
calc_newton_raphson_h.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )

particle_particle_compute_kernel = """

  /* We do not excecute any specific action here */
    
"""

touch_particle_last_time_compute_kernel = """

   if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){

    /* Do a Newton-Raphson iteration.
     * Notice that each iteration requires to re-run the full density loop.
     */
     
    /* Retrieve basic properties of the local particle */
    const double rho_a = localParticle->getDensity();
    const double m_a = localParticle->getMass();  
    const double h_old = localParticle->getSmoothingLength();
    
    const double hydro_eta_dim = std::pow(SMOOTH_L_FAC, Dimensions);
    const double h_old_dim = std::pow(h_old, Dimensions);
    const double h_old_dim_minus_one = std::pow(h_old, Dimensions - 1);
      
    const double dh_drho_a = - h_old/rho_a/Dimensions;
//    const double Omega_a = 1.0 - dh_drho_a * m_a* localParticle->getWcount_dh();
    const double Omega_a = localParticle->getOmega();
     
//    /* Update Omega to use later in force loop calculations */
//    localParticle->setOmega( Omega_a );
   
    /* Value at the start of iteration procedure (last update on previous time-step) */
    const double h_0 = localParticle->getSmoothingLengthPrevious();

//    const double h_0 = localParticle->getSmoothingLength() +
//                         GLOBAL_TIME_STEP_SIZE * localParticle->getHDot(); 

    const double htolerance = 1E-2; // Tolerance for convergence of N-R iterations
       
    // Safety checks before doing NR
    assertion4( not rho_a==0, localParticle->toString(), localParticle->getDensity(), localParticle->getOmega(),rho_a );  
    
    /* The equation that NR solves is 
     *     f = rho_target - rho_sum = 0. 
     * This is the same convention as PySPH, and opposite of Price et al. and Swift
     */
      
    double func = m_a * std::pow( SMOOTH_L_FAC/localParticle->getSmoothingLength(), Dimensions );
    func += -rho_a;
  
    double dfunc_dh = Omega_a/dh_drho_a; // insert (-) if following Price et al convention.
    
    /* Swift method */
    
    const double n_sum = localParticle->getWcount() * h_old_dim;
    const double n_target = hydro_eta_dim;
    const double f = n_sum - n_target;
    const double f_prime = localParticle->getWcount_dh() * h_old_dim +
                              Dimensions * localParticle->getWcount() * h_old_dim_minus_one;

    /* Get new value of smoothing length via N-R iteration */
//    double h_new = localParticle->getSmoothingLength() - f/f_prime;
                             
    /* Get new value of smoothing length via N-R iteration */
    double h_new = localParticle->getSmoothingLength() - func/dfunc_dh;

    /* Safety check: truncate to the range [ h_old/2 , 2h_old ]. */
    h_new = std::min(h_new, 2.0 * h_old);
    h_new = std::max(h_new, 0.5 * h_old);
    
    // Fall back to h_old if it has become too small or Nan
    if(localParticle->getSmoothingLength()<= 1e-5 || std::isnan(localParticle->getSmoothingLength()) ) {
      localParticle->setSmoothingLength(h_old);
    }
    
    // Update value of h
    localParticle->setSmoothingLength( h_new );

    // Check if convergence has been achieved
    const double residual = std::abs(h_new - h_old)/h_0 ;
    
    if(residual <= htolerance){
      localParticle->setHasConverged(true);  
    }
    
    // DEBUG checks
    assertion6( localParticle->getSmoothingLength()>1E-5, localParticle->toString(), localParticle->getSmoothingLength(), func, dfunc_dh, localParticle->getOmega(),residual );
      
  }

"""

touch_particle_first_time_compute_kernel = """

    /* Prepare particles for Newton-Rhapson */
    
    if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){

      localParticle->setHasConverged(false);
      
      // DEBUG
      assertion3( not localParticle->getDensity()==0, localParticle->toString(), localParticle->getDensity(), localParticle->getOmega() );  

    }  
"""

calc_newton_raphson_h.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel=particle_particle_compute_kernel,
  touch_vertex_first_time_compute_particle_update_kernel=touch_particle_first_time_compute_kernel,
  touch_vertex_last_time_compute_particle_update_kernel=touch_particle_last_time_compute_kernel,
  additional_includes="""
#include "Constants.h" 
  """
))
calc_newton_raphson_h.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
calc_newton_raphson_h.add_action_set( particle_tree_analysis )
project.solversteps.add_step(calc_newton_raphson_h)

## hydro_prepare_force

Prepare a particle for the force calculation.

In [12]:
hydro_prepare_force = peano4.solversteps.Step( "HydroPrepareForce" )
hydro_prepare_force.use_vertex(particles)
hydro_prepare_force.use_cell(particle_tree_analysis.cell_marker)
hydro_prepare_force.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
hydro_prepare_force.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
  
  /* We do not excecute any specific action here */  
  
""",

  touch_vertex_first_time_compute_particle_update_kernel="""
 
  if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){

  // Retrieve basic properties of local and active particles
    const double rho_a = localParticle->getDensity();
    const double m_a = localParticle->getMass();  
    const double h_a = localParticle->getSmoothingLength();

    const double h_inv = 1.0/h_a;
  
    /* Compute the norm of the curl */
    /* 2D sims */
    const double curl_v = std::sqrt( localParticle->getRot_v() * localParticle->getRot_v() );
  
    /* Compute the norm of div v */
    const double abs_div_v = tarch::la::abs( localParticle->getDiv_v() );
    
    /* Compute the sound speed */
    const double soundspeed = gas_soundspeed_from_internal_energy( localParticle->getU() );

    // Construct final expression of Omega
    /* Swift Expression for equivalent to Omega term ("f", see also Springel and Hernquist) */
   
    const double dh_drho_a = - h_a / rho_a / Dimensions;
    const double common_factor = h_a / Dimensions / localParticle->getWcount();
    
    double grad_h_term;
    const double grad_W_term = common_factor * localParticle->getWcount_dh();
    
    // Avoid problematic particles
    if(grad_W_term < -0.9999){
         /* if we get here, we either had very small neighbour contributions
           (which should be treated as a no neighbour case in the ghost) or
           a very weird particle distribution (e.g. particles sitting on
           top of each other). Either way, we cannot use the normal
           expression, since that would lead to overflow or excessive round
           off and cause excessively high accelerations in the force loop */
      grad_h_term = 0;
    } else {
      grad_h_term = common_factor * localParticle->getRho_dh()/(1.0 + grad_W_term);
    }

    /* Compute the Balsara Switch */
    const double balsara = ALPHA_AV * abs_div_v /
                           (abs_div_v + curl_v + 0.0001 * soundspeed * h_inv);
                               
    assertion6( std::isfinite(balsara), localParticle->toString(), balsara, abs_div_v, 
                                        curl_v, soundspeed, ALPHA_AV );
  
    /* Update variables */
    const double P_a = gas_pressure_from_internal_energy( rho_a, localParticle->getU() );
    
    /* SWIFT */
//    localParticle->setOmega( grad_h_term );  
    /* Price */
    localParticle->setOmega( 1.0 - dh_drho_a * localParticle->getRho_dh() ); 
    
    localParticle->setPressure(P_a);
    localParticle->setSoundSpeed(soundspeed);
    localParticle->setBalsara(balsara);

  }

""",
  touch_vertex_last_time_compute_particle_update_kernel="""

  /* We do not excecute any specific action here */

""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
#include "kernel_hydro.h"
#include "equation_of_state.h"
"""
  ))
hydro_prepare_force.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
hydro_prepare_force.add_action_set( particle_tree_analysis )
project.solversteps.add_step(hydro_prepare_force)

## Force loop

In [13]:
force_non_sym_particles = peano4.solversteps.Step( "ForceNonSym" )
force_non_sym_particles.use_vertex(particles)
force_non_sym_particles.use_cell(particle_tree_analysis.cell_marker)
force_non_sym_particles.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
force_non_sym_particles.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
  
/*
 * Non-symmetric force interaction between two particles (as per Swift terminology). 
 * In this, only the Local particle is updated (in the symmetric case, both are updated at the same time).
 * Time derivative of the internal energy, du/dt, is also updated here.
 * Subscripts a and b represent indices of Local and Active particles, respectively:
 *
 * h_a: Smoothing length of Local particle
 * h_b: Smoothign length of Active particle
 * dx : Vector separating the two particles (x_loc - x_act)
 * r  : Distance between the two particles  (|| dx ||)
 *
 */
  
  
  /* Distance between the two particles */
  tarch::la::Vector<Dimensions,double> dx = localParticle->getX() - activeParticle->getX();
  const double r = tarch::la::norm2(dx);
  const double r_inv = r ? 1.0 / r : 0.0;

  // Retrieve basic properties of local and active particles
  const double rho_a = localParticle ->getDensity();
  const double rho_b = activeParticle->getDensity();
  
  const double P_a = localParticle ->getPressure();
  const double P_b = activeParticle->getPressure(); 
  
  const double m_a = localParticle ->getMass();  
  const double m_b = activeParticle->getMass();  

  // Smoothing lengths
  const double h_a = localParticle ->getSmoothingLength();
  const double h_b = activeParticle->getSmoothingLength();

  // Dimensionless distance to evaluate kernel
  const double q_a = r / h_a;
  const double q_b = r / h_b;
  
  // Get actual interaction radius
  const double cutoffR = KERNEL_SUPPORT_RAD*h_a;
 
  if ( tarch::la::smaller( r,cutoffR ) ) {
    if ( tarch::la::greater( r,0.0 ) ) {

      // Evaluate kernel and its derivative for particle a
      double w_a, dw_a_dq;
      kernel_deval(q_a, w_a, dw_a_dq);

      // Evaluate kernel and its derivative for particle b
      double w_b, dw_b_dq;
      kernel_deval(q_b, w_b, dw_b_dq);

      // Normalization factors 
      double wa_dr = dw_a_dq / std::pow( h_a, Dimensions + 1 );
      double wb_dr = dw_b_dq / std::pow( h_b, Dimensions + 1 );

      /* Variable smoothing length term */
//      const double f_ab = 1.0 - localParticle ->getOmega()/m_b;
//      const double f_ba = 1.0 - activeParticle->getOmega()/m_a;
      
      const double f_ab = 1.0 / localParticle ->getOmega();
      const double f_ba = 1.0 / activeParticle->getOmega();

      /* Compute gradient terms */
      const double P_over_rho2_a = P_a / (rho_a * rho_a) * f_ab;
      const double P_over_rho2_b = P_b / (rho_b * rho_b) * f_ba;

      /* Compute dv dot r. */
      double dvdr = tarch::la::dot( (localParticle->getV() - activeParticle->getV()), dx );

      /* Are the particles moving towards each others ? */
      const double omega_ab = std::min(dvdr, 0.0);
      const double mu_ab = r_inv * omega_ab; /* This is 0 or negative */
  
      /* Compute sound speeds and signal velocity */
      const double v_sig = localParticle->getSoundSpeed() + activeParticle->getSoundSpeed() -
                           BETA_AV * mu_ab;
  
      /* Balsara term */
      const double balsara_a = localParticle ->getBalsara();
      const double balsara_b = activeParticle->getBalsara();
  
      /* Construct the full viscosity term */
      const double rho_ab = 0.5 * (rho_a + rho_b);
      const double visc = -0.25 * v_sig * mu_ab * (balsara_a + balsara_b) / rho_ab;

      /* Convolve with the kernel */
      const double visc_acc_term = 0.5 * visc * (wa_dr * f_ab + wb_dr * f_ba) * r_inv;

      /* SPH acceleration term */
      const double sph_acc_term = (P_over_rho2_a * wa_dr + P_over_rho2_b * wb_dr) * r_inv;
      
      /* Assemble the acceleration */
      const double acc = sph_acc_term + visc_acc_term;
            
      /* Use the force Luke ! */
      localParticle->setA( localParticle->getA() - m_b * acc * dx );
              
      /* Get the time derivative for u. */
      const double sph_du_term_a = P_over_rho2_a * dvdr * r_inv * wa_dr;

      /* Viscosity term */
      const double visc_du_term = 0.5 * visc_acc_term * dvdr;

      /* Assemble the energy equation term */
      const double du_dt_a = sph_du_term_a + visc_du_term;

      /* Internal energy time derivative */
      localParticle->setUDot(localParticle->getUDot() + m_b * du_dt_a);

      /* Get the time derivative for h. */
      localParticle->setHDot(localParticle->getHDot() - m_b * dvdr * r_inv/rho_b * wa_dr * f_ab);

      /* Update the signal velocity */
      localParticle->setV_sig_AV( std::max(localParticle->getV_sig_AV(), v_sig) );
    }
  }
  
""",

  touch_vertex_first_time_compute_particle_update_kernel="""

  /* Done by hydro_reset_acceleration in Swift */

  // we'll accumulate the data throughout mesh/particle traversal
  
  if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){

    localParticle->setA   (0.0);   
    localParticle->setUDot(0.0);   
  
  }

""",
  touch_vertex_last_time_compute_particle_update_kernel="""

  /* We do not excecute any specific action here */


""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
#include "kernel_hydro.h"
#include "equation_of_state.h"
"""
  ))
force_non_sym_particles.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
force_non_sym_particles.add_action_set( particle_tree_analysis )
project.solversteps.add_step(force_non_sym_particles)

## KDK/leapfrog steps

In the KDK (leapfrog) scheme, we update positions and velocities according to

\begin{align}
{\bf v}^{n+\frac{1}{2}} &= {\bf v}^{n} + {\bf a}^{n}\frac{\Delta t}{2}\\
{\bf x}^{n+1} &= {\bf x}^{n} + {\bf v}^{n+\frac{1}{2}}{\Delta t}\\
{\bf v}^{n+1} &= {\bf v}^{n+\frac{1}{2}} + {\bf a}^{n+1}\frac{\Delta t}{2}\\
\end{align}
 
Notice that the last kick in this sequence is implicit if ${\bf a}$ depends on ${\bf v}$, which requires extra treatments (see Price et al.).


## Kick 1

In [14]:
kick_particles = peano4.solversteps.Step( "Kick" )
kick_particles.use_vertex(particles)
kick_particles.use_cell(particle_tree_analysis.cell_marker)
kick_particles.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
kick_particles.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
    if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
      localParticle->setHasBeenKicked(false);
    }
     
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
  const double timeStepSize_half = 0.5*GLOBAL_TIME_STEP_SIZE;
  
  if (
      not localParticle->getHasBeenKicked()
      and
      localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
    ){
  
    /* Kick the particle */
    
    // Update Velocity
    localParticle->setV( localParticle->getV() + timeStepSize_half * localParticle->getA()    );
    
    /* If predictor is going to be used, store old acceleration */
    localParticle->setA_prev_step( localParticle->getA() );

    assertion1(  std::isfinite(localParticle->getV(0)), localParticle->toString() );
    assertion1(  std::isfinite(localParticle->getV(1)), localParticle->toString() );
    
    localParticle->setHasBeenKicked(true);
    
    }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
kick_particles.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
kick_particles.add_action_set( particle_tree_analysis )
project.solversteps.add_step(kick_particles)

## Kick 2

Same as kick 1 but does not update the accelation array of previous time step

In [15]:
kick2_particles = peano4.solversteps.Step( "Kick2" )
kick2_particles.use_vertex(particles)
kick2_particles.use_cell(particle_tree_analysis.cell_marker)
kick2_particles.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
kick2_particles.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
    if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
      localParticle->setHasBeenKicked(false);
    }
    
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
  const double timeStepSize_half = 0.5*GLOBAL_TIME_STEP_SIZE;
  
  if (
      not localParticle->getHasBeenKicked()
      and
      localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
    ){
  
    /* Kick the particle */
    
    // Update Velocity
    localParticle->setV( localParticle->getV() + timeStepSize_half * localParticle->getA()    );
    
    assertion1(  std::isfinite(localParticle->getV(0)), localParticle->toString() );
    assertion1(  std::isfinite(localParticle->getV(1)), localParticle->toString() );
    
    localParticle->setHasBeenKicked(true);
    
    /* Store value of h at the end of time-step */
    localParticle->setSmoothingLengthPrevious( localParticle->getSmoothingLength() );
    
    }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
kick2_particles.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
kick2_particles.add_action_set( particle_tree_analysis )
project.solversteps.add_step(kick2_particles)

## Drift step

In this step, we update the particle's position via

\begin{align}
{\bf x}^{n+1} = {\bf x}^{n} + {\bf v}^{n+1/2}{\Delta t}
\end{align}

In [16]:
drift_particles = peano4.solversteps.Step( "Drift" )
drift_particles.use_vertex(particles)
drift_particles.use_cell(particle_tree_analysis.cell_marker)
drift_particles.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
drift_particles.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
   if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
     localParticle->setMoveState( globaldata::Particle::MoveState::NotMovedYet );
   }

""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
  const double timeStepSize = GLOBAL_TIME_STEP_SIZE;
  
  if (
    localParticle->getMoveState()!=globaldata::Particle::MoveState::Moved
    and
    localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
    ){
  
    // Drift the particle
    
    // Update Position
    localParticle->setX( localParticle->getX() + timeStepSize * localParticle->getV() );
    
    // Boundary Conditions
    for (int d=0; d<Dimensions; d++) {
      if ( localParticle->getX()(d)<0.0 ) {

        /* Reflecting BC */   
        localParticle->setX( d, std::abs(localParticle->getX()(d)) );
        localParticle->setV( d, std::abs(localParticle->getV()(d)) );

      }
      if ( localParticle->getX()(d)>1.0 ) {

        /* Reflecting BC */   
        localParticle->setX( d, 2.0 - localParticle->getX()(d) );
        localParticle->setV( d,-std::abs(localParticle->getV()(d)) );

      }
    
      localParticle->setMoveState( globaldata::Particle::MoveState::Moved );
      
      } /* End for dimensions */
      
    }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
drift_particles.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
drift_particles.add_action_set( particle_tree_analysis )
project.solversteps.add_step(drift_particles)

## Hydro_kick_extra

In [17]:
hydro_kick_extra = peano4.solversteps.Step( "HydroKickExtra" )
hydro_kick_extra.use_vertex(particles)
hydro_kick_extra.use_cell(particle_tree_analysis.cell_marker)
hydro_kick_extra.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
hydro_kick_extra.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
  /* We do not excecute any specific action here */
     
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
/**
 * @brief Kick the additional variables
 *
 * Additional hydrodynamic quantites are kicked forward in time here. These
 * include thermal quantities (thermal energy or total energy or entropy, ...).
 *
 */

  const double dt_therm = 0.5*GLOBAL_TIME_STEP_SIZE;
  
  if (localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
  
    /* Integrate the internal energy forward in time */
    const double delta_u = localParticle->getUDot() * dt_therm;
  
    /* Do not decrease the energy by more than a factor of 2 */
    localParticle->setU( std::max( localParticle->getU() + delta_u,
                                   localParticle->getU() * 0.5      ) );

    /* Check against absolute minimum.
     * Default value is zero. */
    if (localParticle->getU() < 0.0) {
      localParticle->setU   (0.0);
      localParticle->setUDot(0.0);
    }
    
  }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
hydro_kick_extra.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
hydro_kick_extra.add_action_set( particle_tree_analysis )
project.solversteps.add_step(hydro_kick_extra)

## Corrector step (mini-kick) (Phantom code implementation)

In [18]:
kick_particles_corrector = peano4.solversteps.Step( "KickCorrector" )
kick_particles_corrector.use_vertex(particles)
kick_particles_corrector.use_cell(particle_tree_analysis.cell_marker)
kick_particles_corrector.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
kick_particles_corrector.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
    if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
      localParticle->setHasBeenKicked(false);
    }
    
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
  const double timeStepSize_half = 0.5*GLOBAL_TIME_STEP_SIZE;
  
  if( not localParticle->getHasBeenKicked()
      and
      localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
 ){
  
    /* Kick the particle */
    
    localParticle->setV( localParticle->getV() + 
                         timeStepSize_half * (localParticle->getA() - localParticle->getA_prev_step()) );

    localParticle->setHasBeenKicked(true);

    /* Store old acceleration for a potential additional corrector step */
    localParticle->setA_prev_step( localParticle->getA() );  
    
//    /* Store value of h at the end of time-step */
//    localParticle->setSmoothingLengthPrevious( localParticle->getSmoothingLength() );
    
  }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
kick_particles_corrector.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
kick_particles_corrector.add_action_set( particle_tree_analysis )
project.solversteps.add_step(kick_particles_corrector)

## Corrector for Extra (internal energy)

In [19]:
kick_corrector_extra = peano4.solversteps.Step( "KickCorrectorExtra" )
kick_corrector_extra.use_vertex(particles)
kick_corrector_extra.use_cell(particle_tree_analysis.cell_marker)
kick_corrector_extra.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
kick_corrector_extra.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
    if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
      localParticle->setHasBeenKicked(false);
    }
    
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
  const double timeStepSize_half = 0.5*GLOBAL_TIME_STEP_SIZE;
  
  if(
      not localParticle->getHasBeenKicked()
      and
      localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
 ){
  
    /* Kick the internal energy of the particle */
    
    localParticle->setU( localParticle->getU() + 
                         timeStepSize_half * (localParticle->getUDot() - localParticle->getUDot_prev_step()) );

    localParticle->setHasBeenKicked(true);

    /* Store old value for a potential additional corrector step */
    localParticle->setUDot_prev_step( localParticle->getUDot() );  
    
    
  }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
kick_corrector_extra.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
kick_corrector_extra.add_action_set( particle_tree_analysis )
project.solversteps.add_step(kick_corrector_extra)

## End time-step
Actions executed once at the end of the time-step

In [20]:
end_time_step = peano4.solversteps.Step( "EndTimeStep" )
end_time_step.use_vertex(particles)
end_time_step.use_cell(particle_tree_analysis.cell_marker)
end_time_step.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
end_time_step.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
    /* Mark particle needs to be updated */

    if(localParticle->getParallelState()==globaldata::Particle::ParallelState::Local){
      localParticle->setHasBeenKicked(false);
    }
    
""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
  const double timeStepSize = GLOBAL_TIME_STEP_SIZE;
  
  if( not localParticle->getHasBeenKicked()
      and
      localParticle->getParallelState()==globaldata::Particle::ParallelState::Local
 ){
  
    /* Record elapsed time */
    localParticle->setTime( localParticle->getTime() + timeStepSize );
    
    /* Calculate max time-step size value allowed by the CFL condition */
    const double soundspeed = localParticle->getSoundSpeed();
    const double timeStepSizeCFL = CFL_CONSTANT * localParticle->getSmoothingLength()*KERNEL_SUPPORT_RAD/soundspeed;
    localParticle->setTimeStepSize(timeStepSizeCFL);

    /* Store value of h at the end of time-step */
    localParticle->setSmoothingLengthPrevious( localParticle->getSmoothingLength() );
    
    /* Mark particle is done */
    localParticle->setHasBeenKicked(true);
    
  }
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
end_time_step.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
end_time_step.add_action_set( particle_tree_analysis )
project.solversteps.add_step(end_time_step)

## For Debugging: collapse problem into 1D

This is an initial, extra step to move particles into a straight line when the simulation starts.

In [21]:
rearrange_particles = peano4.solversteps.Step( "RearrangeAtStart" )
rearrange_particles.use_vertex(particles)
rearrange_particles.use_cell(particle_tree_analysis.cell_marker)
rearrange_particles.add_action_set( peano4.toolbox.particles.UpdateParticleGridAssociation(particles) )
rearrange_particles.add_action_set( peano4.toolbox.particles.ParticleParticleInteraction(
  particle_set=particles,
  particle_particle_interaction_kernel="""
    
  /* We do not excecute any specific action here */
  
""",
  touch_vertex_first_time_compute_particle_update_kernel="""
  
  /* We do not excecute any specific action here */

""",
  touch_vertex_last_time_compute_particle_update_kernel="""
  
/*
  // 1D test -- put particle into straight lines

  if( localParticle->getX(1)>0.51 ){
    localParticle->setX(1, 0.99);
  }else if(localParticle->getX(1)<0.49){
    localParticle->setX(1, 0.01);
  }
  
*/  


/*
  if( localParticle->getX(0)<1e-2 ){
    localParticle->setX(0, -5);
  } else if( localParticle->getX(1)<1e-2 ){
    localParticle->setX(1, -5);
  }  
*/  
  
    
""",
  additional_includes="""
#include "Constants.h"
#include <cmath>
"""
  ))
rearrange_particles.add_action_set( peano4.toolbox.particles.ParticleAMR(particles,particle_tree_analysis,min_particles_per_cell=10) )
rearrange_particles.add_action_set( particle_tree_analysis )
project.solversteps.add_step(rearrange_particles)

### Plots setup

I will have to plot the outcome (or any snapshot). With particles, each of my dumps produces two types of files: particle data and mesh data. For the mesh, I rely on the toolbox's grid plotting. The plot is the step where we don't need the AMR criterion - the particles don't move here after all. We however run the tree analysis, so it is up-to-date for the next step.

In [22]:
# Particle plotter
particle_plotter = peano4.toolbox.particles.PlotParticlesInVTKFormat( "particles", particles, time_stamp_evaluation=peano4.toolbox.particles.PlotParticlesInVTKFormat.CountTimeSteps )
particle_plotter.add_attribute_to_plot(v_attr       , Dimensions)
particle_plotter.add_attribute_to_plot(a_attr       , Dimensions)
particle_plotter.add_attribute_to_plot(a_prev_step_attr, Dimensions)
particle_plotter.add_attribute_to_plot(density_attr , 1)
particle_plotter.add_attribute_to_plot(pressure_attr, 1)
particle_plotter.add_attribute_to_plot(u_attr       , 1)
particle_plotter.add_attribute_to_plot(mass_attr    , 1)
particle_plotter.add_attribute_to_plot(smoothL_attr , 1)
particle_plotter.add_attribute_to_plot(smoothL_prev_attr , 1)
particle_plotter.add_attribute_to_plot(Omega_attr ,  1)
particle_plotter.add_attribute_to_plot(rho_dh_attr,  1)
particle_plotter.add_attribute_to_plot(Ncount_attr,  1)
particle_plotter.add_attribute_to_plot(wcount_attr,  1)
particle_plotter.add_attribute_to_plot(wcount_dh_attr,  1)
particle_plotter.add_attribute_to_plot(soundSpeed_attr, 1)
particle_plotter.add_attribute_to_plot(has_converged_attr, 1)
particle_plotter.add_attribute_to_plot(time_attr, 1)
particle_plotter.add_attribute_to_plot(timeStepSize_attr, 1)

# Artificial viscosity
particle_plotter.add_attribute_to_plot(rot_v_attr   , 1)
particle_plotter.add_attribute_to_plot(div_v_attr   , 1)
particle_plotter.add_attribute_to_plot(v_sig_AV_attr, 1)
particle_plotter.add_attribute_to_plot(balsara_attr, 1)

# Solution plotter
print_solution = peano4.solversteps.Step( "Plot", add_user_defined_actions=False )
print_solution.use_vertex(particles)
print_solution.use_cell(particle_tree_analysis.cell_marker)
print_solution.add_action_set( particle_tree_analysis )
print_solution.add_action_set( peano4.toolbox.PlotGridInPeanoBlockFormat( filename="grid", time_stamp_evaluation=peano4.toolbox.PlotGridInPeanoBlockFormat.CountTimeSteps ) )
print_solution.add_action_set( particle_plotter )

project.solversteps.add_step(print_solution)

## Generate the actual C++ code

Next, we generate the actual Peano 4 C++ code:

In [23]:
project.generate()

generate all code ...
user has to modify class DensityNonSym in actions directory manually 
user has to modify class NewtonRaphsonSmoothingLength in actions directory manually 
user has to modify class HydroPrepareForce in actions directory manually 
user has to modify class ForceNonSym in actions directory manually 
user has to modify class Kick in actions directory manually 
user has to modify class Kick2 in actions directory manually 
user has to modify class Drift in actions directory manually 
user has to modify class HydroKickExtra in actions directory manually 
user has to modify class KickCorrector in actions directory manually 
user has to modify class KickCorrectorExtra in actions directory manually 
user has to modify class EndTimeStep in actions directory manually 
user has to modify class RearrangeAtStart in actions directory manually 
generated minimal-sph-peano-test-sedov-main.cpp
write ./Makefile
./vertexdata/ParticleSet.h written by Jinja2TemplatedHeaderImplementationF

write ././observers/Kick2.h
././observers/Kick2.cpp written by Observer.py
././actions/Drift.h written by ActionSet.py
././actions/Drift.cpp written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_UpdateParticleGridAssociation1.h written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_UpdateParticleGridAssociation1.cpp written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_ParticleParticleInteraction2.h written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_ParticleParticleInteraction2.cpp written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_ParticleAMR3.h written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_ParticleAMR3.cpp written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_ParticleTreeAnalysis4.h written by ActionSet.py
././observers/Drift2peano4_toolbox_particles_ParticleTreeAnalysis4.cpp written by ActionSet.py
write ././observers/Drift.h
././observers/Drift.cpp written by Observ

## Write the main

Do not forget to include the generated header files, e.g., 
    
    //
    // =================
    // inserted manually
    // =================
    //
    #include "observers/CreateGrid.h"
    #include "observers/Init.h"
    #include "observers/Plot.h"
    #include "observers/Move.h"


## Build the code

In [24]:
project.build()

clean up project ...
rm -f celldata/ParticleSetCellStatistics.o vertexdata/ParticleSet.o globaldata/Particle.o repositories/DataRepository.o ./observers/CreateGrid2peano4_toolbox_CreateRegularGrid0.o ./observers/CreateGrid.o ./observers/Init2peano4_toolbox_particles_UpdateParticleGridAssociation0.o ./observers/Init2peano4_toolbox_particles_InsertParticlesAlongCartesianLayoutIntoUnrefinedCells1.o ./observers/Init2peano4_toolbox_particles_ParticleTreeAnalysis2.o ./observers/Init.o ./actions/DensityNonSym.o ./observers/DensityNonSym2peano4_toolbox_particles_UpdateParticleGridAssociation1.o ./observers/DensityNonSym2peano4_toolbox_particles_ParticleParticleInteraction2.o ./observers/DensityNonSym2peano4_toolbox_particles_ParticleAMR3.o ./observers/DensityNonSym2peano4_toolbox_particles_ParticleTreeAnalysis4.o ./observers/DensityNonSym.o ./actions/NewtonRaphsonSmoothingLength.o ./observers/NewtonRaphsonSmoothingLength2peano4_toolbox_particles_UpdateParticleGridAssociation1.o ./observers/New

## Run code

In [25]:
# remove output files from previous runs
output_files = [ f for f in os.listdir(".") if f.endswith(".peano-patch-file") or f.endswith(".vtu") or f.endswith(".pvd") ]
for f in output_files:
  os.remove(f)

# success = project.run( args=["--threads", "1"], prefix=["mpirun", "-n", "1"] )
#!./peano4

----------------------

# Summary of SPH equations
We follow the review paper by Price et al (2018).

Notice that, throughout this, $h$ denotes the smoothing length.

#### SPH basics

\begin{align}
\rho_a& = \sum_b m_b W(|{\bf r}_a - {\bf r}_b|, h_a)\\
\delta\Omega_a\equiv\Omega_a &= 1 - \frac{\partial h_a}{\partial \rho_a}\sum_b m_b\frac{\partial W_{ab}(h_a)}{\partial h_a} \qquad\qquad (\delta\Omega_a=0 \quad \textrm{for constant $h$})\\
\frac{{\rm d}{\rho_a}}{{\rm d}t} &= \frac{1}{\Omega_a}\sum_b m_b{v}_{ab}\cdot{\bf \nabla}_aW_{ab}(h_a)
\end{align}

#### Equation of State (EoS) et al

\begin{align}
P &=(\gamma-1)\rho u \qquad \textrm{Equation of State (EoS)}
\end{align}



#### Euler equation (w/o artificial viscosity terms)

\begin{align}
\frac{{\rm d}{\bf v}_a}{{\rm d}t} &=-\sum_b m_b\left[\frac{P_a}{\rho_a^2\Omega_a}{\bf \nabla}_aW_{ab}(h_a) + \frac{P_b}{\rho_b^2\Omega_b}{\bf \nabla}_aW_{ab}(h_b) \right] + {\bf a}^{\rm self-grav}_a + {\bf a}^{\rm others}_a\\
\end{align}


#### Evolution of internal energy:

\begin{align}
\frac{{\rm d}u_a}{{\rm d}t} &=\frac{P_a}{\rho_a^2\Omega_a}\sum_b m_b{v}_{ab}\cdot{\bf \nabla}_aW_{ab}(h_a)\\
\end{align}


#### Newton-Raphson calculation of $\rho$ and (variable) $h$:

\begin{align}
\zeta(h_a)&\equiv m_a\left(\frac{\eta}{h_a}\right)^{3}-\sum_b m_b W_{ab}(h_a)=0\\
\textrm{Initial guess:} &\qquad\qquad h^{\rm ini}_a =h_a + \Delta t \frac{\partial h_a}{\partial\rho_a}\frac{d\rho_a}{dt}\\ 
\textrm{N-R iteration:}&\qquad\qquad h_{a,{\rm new}}=h_a - \frac{\zeta({h_a})}{\zeta'(h_a)} \qquad \textrm{where:}\qquad \zeta'(h_a)=-\frac{3\rho_a \Omega_a}{h_a} \\
\end{align}
 

#### Useful relations

\begin{align}
\nabla_aW_{ab}&=\frac{C_{\rm norm}}{h^4}f'(q)\hat{{\bf r}}_{ab} \\
\frac{\partial W_{ab}}{\partial h}(r,h)&=-\frac{C_{\rm norm}}{h^4}[3f(q)+qf'(q)] \\
\frac{\partial h_a}{\partial \rho_a} &= -\frac{3h_a}{\rho_a} \\
\end{align}